# RemoteSea AI — CANDIDATE_ANALYSIS QLoRA Training (Google Colab)

**This notebook must run on a Colab GPU runtime, never on the local RemoteSea AI Service machine.**
The `remotesea-ai` FastAPI service and its Postgres database run on a GPU-less
Intel Mac by design — training happens here, on Colab, as a fully separate
pipeline. Nothing in this notebook talks to the local machine directly; you
manually move files (the dataset in, the trained adapter out).

Before running: **Runtime → Change runtime type → T4 GPU** (or better).

## What this notebook does

1. Loads `train.jsonl` / `validation.jsonl` / `test.jsonl` produced by
   `POST /v1/datasets/build` on the AI Service (see that service's README).
2. Fine-tunes `Qwen/Qwen2.5-3B-Instruct` with QLoRA (4-bit base + LoRA
   adapter) on the `CANDIDATE_ANALYSIS` task only.
3. Evaluates the fine-tuned model against `test.jsonl` (JSON validity,
   recommendation accuracy, score error).
4. Saves the LoRA adapter so it can be downloaded and registered with the
   AI Service's Model Registry (`POST /v1/models`, `/complete-training`,
   `/evaluate`) from your local machine.

**Disclosure**: this notebook has been written but not executed — the
environment building it has no GPU. Run it end-to-end on Colab yourself
before trusting its output; treat the hyperparameters below as reasonable
T4-friendly defaults, not values verified against a real training run.


## 1. Install dependencies

In [ ]:
!pip install -q -U transformers accelerate peft bitsandbytes trl datasets sentencepiece


## 2. Verify GPU

Fails fast with a clear message rather than silently falling back to a CPU
run that would take hours instead of minutes.


In [ ]:
import torch

assert torch.cuda.is_available(), (
    "No CUDA GPU visible. Go to Runtime -> Change runtime type -> select a "
    "GPU (T4 or better), then Runtime -> Restart session."
)
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA capability:", torch.cuda.get_device_capability(0))


## 3. Configuration

Conservative defaults for a single T4 (16GB VRAM): batch size 1 with
gradient accumulation, a fairly short max sequence length (CANDIDATE_ANALYSIS
inputs/outputs are compact JSON, not long documents), and 4-bit base model
weights via bitsandbytes.


In [ ]:
BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"

# Point these at wherever you uploaded the Dataset Builder's output for the
# dataset version you're training on (see the AI Service README's
# "Verify feedback + dataset build" section for how that's produced).
TRAIN_PATH = "train.jsonl"
VALIDATION_PATH = "validation.jsonl"
TEST_PATH = "test.jsonl"

OUTPUT_DIR = "candidate_analysis_lora"
MAX_SEQ_LENGTH = 1024

LORA_CONFIG = dict(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    # Qwen2 attention + MLP projection layer names.
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

TRAINING_ARGS = dict(
    output_dir=f"{OUTPUT_DIR}/checkpoints",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    report_to=[],
)


## 4. Upload the dataset

Run this cell and pick `train.jsonl`, `validation.jsonl`, `test.jsonl` from
the `datasets/candidate_analysis/v{N}/` directory the AI Service wrote them
to. (If `validation.jsonl` / `test.jsonl` are empty — the Dataset Builder
leaves them empty when there are under 10 examples — training still works,
just without held-out evaluation until the dataset grows.)


In [ ]:
from google.colab import files

uploaded = files.upload()


## 5. Load and format the dataset

In [ ]:
import json
from datasets import load_dataset

def _load_jsonl_or_empty(path):
    try:
        with open(path) as f:
            lines = [line for line in f if line.strip()]
        if not lines:
            return None
    except FileNotFoundError:
        return None
    return load_dataset("json", data_files=path, split="train")

train_dataset = _load_jsonl_or_empty(TRAIN_PATH)
validation_dataset = _load_jsonl_or_empty(VALIDATION_PATH)
test_dataset = _load_jsonl_or_empty(TEST_PATH)

assert train_dataset is not None, f"{TRAIN_PATH} is missing or empty — nothing to train on."
print(f"train: {len(train_dataset)} examples")
print(f"validation: {len(validation_dataset) if validation_dataset else 0} examples")
print(f"test: {len(test_dataset) if test_dataset else 0} examples")


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token

def format_example(example):
    # Each row is {"instruction": ..., "input": {...}, "output": {...}} —
    # exactly what app/services/dataset_builder.py writes. The instruction
    # is the same system prompt CANDIDATE_ANALYSIS is served with at
    # inference time (see inference_service.py's _CANDIDATE_ANALYSIS_SYSTEM)
    # — training on a different phrasing than what's used in production
    # would defeat the point.
    messages = [
        {"role": "system", "content": example["instruction"]},
        {"role": "user", "content": json.dumps(example["input"], ensure_ascii=False)},
        {"role": "assistant", "content": json.dumps(example["output"], ensure_ascii=False)},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False)

def formatting_func(batch):
    return [format_example(ex) for ex in batch]


## 6. Load the 4-bit base model and attach LoRA

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
)
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(**LORA_CONFIG)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()


## 7. Train

In [ ]:
from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(max_seq_length=MAX_SEQ_LENGTH, **TRAINING_ARGS)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    formatting_func=formatting_func,
)

trainer.train()


## 8. Save the LoRA adapter

Only the adapter is saved (a few tens of MB), not the merged full model —
the AI Service's Model Registry stores this path (`artifactPath`) and, if
you're deploying to Ollama, `scripts/deploy_to_ollama.sh` merges it with the
base model at deploy time (see that script's own disclosure — it hasn't
been run against a real adapter either, for the same no-GPU reason).


In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

import shutil
shutil.make_archive(OUTPUT_DIR, "zip", OUTPUT_DIR)
print(f"Saved adapter to {OUTPUT_DIR}/, zipped to {OUTPUT_DIR}.zip")

from google.colab import files as colab_files
colab_files.download(f"{OUTPUT_DIR}.zip")


## 9. Evaluate against `test.jsonl`

Computes exactly the kind of metrics dict `POST /v1/models/{id}/evaluate`
expects (`{"metrics": {...}, "passed": true|false}`) — see the AI Service
README's model registry section. `passed` is a judgment call for you to
make from these numbers (e.g. `schemaValidRate` above some bar and
`recommendationAccuracy` beating the currently-ACTIVE model), not something
this notebook decides on its own — same reasoning as
`ModelRegistryService.submit_evaluation` taking `passed` as an explicit
input rather than computing it from a hardcoded threshold.


In [ ]:
VALID_RECOMMENDATIONS = {"SHORTLIST", "REVIEW", "REJECT"}

def is_valid_candidate_analysis(output) -> bool:
    if not isinstance(output, dict):
        return False
    if not isinstance(output.get("score"), (int, float)) or not (0 <= output["score"] <= 100):
        return False
    if output.get("recommendation") not in VALID_RECOMMENDATIONS:
        return False
    for key in ("strengths", "weaknesses", "missingRequirements"):
        if not isinstance(output.get(key), list):
            return False
    if not isinstance(output.get("reasoning"), str):
        return False
    return True

def generate(example) -> dict | None:
    messages = [
        {"role": "system", "content": example["instruction"]},
        {"role": "user", "content": json.dumps(example["input"], ensure_ascii=False)},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output_ids = model.generate(**inputs, max_new_tokens=400, do_sample=False)
    raw = tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    start, end = raw.find("{"), raw.rfind("}")
    if start == -1 or end == -1 or end < start:
        return None
    try:
        return json.loads(raw[start : end + 1])
    except json.JSONDecodeError:
        return None


def evaluate(dataset) -> dict:
    total = len(dataset)
    schema_valid = 0
    recommendation_matches = 0
    score_errors = []

    for example in dataset:
        predicted = generate(example)
        ground_truth = example["output"]
        if predicted is not None and is_valid_candidate_analysis(predicted):
            schema_valid += 1
            if predicted["recommendation"] == ground_truth["recommendation"]:
                recommendation_matches += 1
            score_errors.append(abs(predicted["score"] - ground_truth["score"]))

    return {
        "sampleCount": total,
        "schemaValidRate": schema_valid / total if total else 0.0,
        "recommendationAccuracy": recommendation_matches / total if total else 0.0,
        "meanAbsoluteScoreError": sum(score_errors) / len(score_errors) if score_errors else None,
    }


if test_dataset:
    metrics = evaluate(test_dataset)
else:
    print("test.jsonl was empty — evaluating against validation.jsonl instead.")
    metrics = evaluate(validation_dataset) if validation_dataset else {}

print(json.dumps(metrics, indent=2))


## 10. Register the result with the AI Service's Model Registry

Run this from your **local machine** (not Colab — it needs to reach the AI
Service, which runs on your laptop), after downloading the adapter zip from
step 8. This is the exact sequence the Model Registry's README section
documents: register → complete-training → evaluate → (you review the
metrics and) approve.


In [ ]:
# Reference only — fill in real values and run locally, not in this Colab
# session. Left unexecuted here for the same reason as the rest of this
# notebook: no path from this environment to a running AI Service instance.

# import requests
#
# BASE_URL = "http://localhost:8000"
# SECRET = "<AI_SERVICE_SECRET from remotesea-ai/.env>"
# HEADERS = {"Authorization": f"Bearer {SECRET}"}
#
# model = requests.post(f"{BASE_URL}/v1/models", headers=HEADERS, json={
#     "task": "CANDIDATE_ANALYSIS",
#     "name": "candidate-analysis-lora-v1",
#     "baseModel": "Qwen/Qwen2.5-3B-Instruct",
#     "datasetVersion": 1,  # the AiDataset.version you trained on
# }).json()
#
# requests.post(
#     f"{BASE_URL}/v1/models/{model['id']}/complete-training",
#     headers=HEADERS,
#     json={"artifactPath": "/local/path/to/candidate_analysis_lora"},
# )
#
# requests.post(
#     f"{BASE_URL}/v1/models/{model['id']}/evaluate",
#     headers=HEADERS,
#     json={"metrics": metrics, "passed": True},  # `metrics` from step 9
# )
#
# # Only after a human reviews the metrics above:
# # requests.post(f"{BASE_URL}/v1/models/{model['id']}/approve", headers=HEADERS)
